In [1]:
import pandas as pd

# Load golden dataset
golden_folder = "golden_dataset"

data = {}

import os

for file in os.listdir(golden_folder):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        data[name] = pd.read_csv(os.path.join(golden_folder, file))

print("Golden dataset loaded successfully.")
print("Total tables:", len(data))

Golden dataset loaded successfully.
Total tables: 18


In [2]:
# Check recovery by risk segment

accounts = data["accounts"].copy()
payments = data["payments"].copy()

successful_payments = payments[
    payments["payment_status"] == "SUCCESS"
][["account_id"]].drop_duplicates()

risk_recovery = accounts[["account_id", "risk_segment"]].drop_duplicates()

risk_recovery["paid"] = risk_recovery["account_id"].isin(
    successful_payments["account_id"]
)

risk_summary = (
    risk_recovery.groupby("risk_segment")
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("paid", "sum")
    )
    .reset_index()
)

risk_summary["recovery_rate"] = (
    risk_summary["paid_accounts"]
    / risk_summary["total_accounts"]
    * 100
).round(2)

risk_summary

,risk_segment,total_accounts,paid_accounts,recovery_rate
0,HIGH,7552,3300,43.70
1,LOW,7513,3381,45.00
2,MEDIUM,7533,3335,44.27
3,NPA,7402,3268,44.15


In [3]:
# Check risk segment values

accounts["risk_segment"].value_counts()

risk_segment
HIGH      7552
MEDIUM    7533
LOW       7513
NPA       7402
Name: count, dtype: int64

In [4]:
# Create account cohorts

accounts["opened_at"] = pd.to_datetime(accounts["opened_at"])
accounts["cohort_month"] = accounts["opened_at"].dt.to_period("M")

accounts[["account_id", "opened_at", "cohort_month"]].head()

,account_id,opened_at,cohort_month
0,ACC0000001,2025-11-11 04:37:00,2025-11
1,ACC0000002,2025-11-13 15:59:44,2025-11
2,ACC0000003,2025-09-12 04:59:20,2025-09
3,ACC0000004,2025-05-25 19:29:38,2025-05
4,ACC0000005,2025-09-07 14:59:13,2025-09


In [5]:
# Calculate recovery by cohort

successful_accounts = (
    payments[payments["payment_status"] == "SUCCESS"]
    [["account_id"]]
    .drop_duplicates()
)

cohort_data = accounts[["account_id", "cohort_month"]].copy()

cohort_data["paid"] = cohort_data["account_id"].isin(
    successful_accounts["account_id"]
)

cohort_summary = (
    cohort_data.groupby("cohort_month")
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("paid", "sum")
    )
    .reset_index()
)

cohort_summary["recovery_rate"] = (
    cohort_summary["paid_accounts"]
    / cohort_summary["total_accounts"]
    * 100
).round(2)

cohort_summary

,cohort_month,total_accounts,paid_accounts,recovery_rate
0,2024-01,1311,551,42.03
1,2024-02,1299,586,45.11
2,2024-03,1404,604,43.02
3,2024-04,1263,569,45.05
4,2024-05,1287,556,43.20
5,2024-06,1219,546,44.79
6,2024-07,1324,621,46.90
7,2024-08,1358,608,44.77
8,2024-09,1326,552,41.63
9,2024-10,1362,621,45.59


In [6]:
# Check targeting by risk segment

targeting = data["daily_targeting"].copy()

targeting_accounts = targeting.merge(
    accounts[["account_id", "risk_segment"]],
    on="account_id",
    how="left"
)

selection_summary = (
    targeting_accounts.groupby("risk_segment")
    .size()
    .reset_index(name="targeted_records")
)

selection_summary

,risk_segment,targeted_records
0,HIGH,11225
1,LOW,11350
2,MEDIUM,11414
3,NPA,11011


In [7]:
# Compare targeted and non-targeted accounts

targeted_accounts = targeting["account_id"].drop_duplicates()

selection_data = accounts[["account_id", "risk_segment"]].copy()

selection_data["targeted"] = selection_data["account_id"].isin(
    targeted_accounts
)

selection_data["paid"] = selection_data["account_id"].isin(
    successful_accounts["account_id"]
)

selection_summary = (
    selection_data.groupby("targeted")
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("paid", "sum")
    )
    .reset_index()
)

selection_summary["recovery_rate"] = (
    selection_summary["paid_accounts"]
    / selection_summary["total_accounts"]
    * 100
).round(2)

selection_summary

,targeted,total_accounts,paid_accounts,recovery_rate
0,False,6656,2996,45.01
1,True,23344,10288,44.07


In [8]:
# Check account survival across the analysis period

accounts["opened_at"] = pd.to_datetime(accounts["opened_at"])

last_date = payments["event_at"].max()
last_date = pd.to_datetime(last_date)

accounts["survived"] = accounts["opened_at"] <= last_date

survival_summary = (
    accounts.groupby("survived")
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("account_id", lambda x: x.isin(successful_accounts["account_id"]).sum())
    )
    .reset_index()
)

survival_summary["recovery_rate"] = (
    survival_summary["paid_accounts"]
    / survival_summary["total_accounts"]
    * 100
).round(2)

survival_summary

,survived,total_accounts,paid_accounts,recovery_rate
0,True,30000,13284,44.28


In [9]:
# Check final account status

status_history = data["account_status_history"].copy()
status_history["event_at"] = pd.to_datetime(status_history["event_at"])

latest_status = (
    status_history.sort_values("event_at")
    .groupby("account_id")
    .tail(1)
)

latest_status["status"].value_counts()

status
PAID          3760
NPA           3749
WRITEOFF      3729
CLOSED        3723
PTP           3707
ACTIVE        3690
DELINQUENT    3641
Name: count, dtype: int64

In [10]:
# Compare recovery by final account status

survival_data = latest_status[["account_id", "status"]].copy()

survival_data["paid"] = survival_data["account_id"].isin(
    successful_accounts["account_id"]
)

survival_summary = (
    survival_data.groupby("status")
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("paid", "sum")
    )
    .reset_index()
)

survival_summary["recovery_rate"] = (
    survival_summary["paid_accounts"]
    / survival_summary["total_accounts"]
    * 100
).round(2)

survival_summary

,status,total_accounts,paid_accounts,recovery_rate
0,ACTIVE,3690,1663,45.07
1,CLOSED,3723,1641,44.08
2,DELINQUENT,3641,1600,43.94
3,NPA,3749,1599,42.65
4,PAID,3760,1691,44.97
5,PTP,3707,1656,44.67
6,WRITEOFF,3729,1674,44.89


In [11]:
# Check recovery by month and risk segment

accounts["opened_at"] = pd.to_datetime(accounts["opened_at"])
payments["event_at"] = pd.to_datetime(payments["event_at"])

accounts["month"] = accounts["opened_at"].dt.to_period("M").astype(str)

monthly_risk = (
    accounts[["account_id", "risk_segment", "month"]]
    .drop_duplicates()
)

monthly_risk["paid"] = monthly_risk["account_id"].isin(
    successful_accounts["account_id"]
)

simpson_summary = (
    monthly_risk.groupby(["month", "risk_segment"])
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("paid", "sum")
    )
    .reset_index()
)

simpson_summary["recovery_rate"] = (
    simpson_summary["paid_accounts"]
    / simpson_summary["total_accounts"]
    * 100
).round(2)

simpson_summary

,month,risk_segment,total_accounts,paid_accounts,recovery_rate
0,2024-01,HIGH,317,135,42.59
1,2024-01,LOW,342,139,40.64
2,2024-01,MEDIUM,345,150,43.48
3,2024-01,NPA,307,127,41.37
4,2024-02,HIGH,314,134,42.68
...,...,...,...,...,...
87,2025-10,NPA,337,152,45.10
88,2025-11,HIGH,307,131,42.67
89,2025-11,LOW,334,150,44.91
90,2025-11,MEDIUM,308,142,46.10


In [12]:
# Compare overall and segment recovery by month

payments["month"] = payments["event_at"].dt.to_period("M").astype(str)

monthly_accounts = accounts[
    ["account_id", "risk_segment"]
].drop_duplicates()

successful_monthly = (
    payments[payments["payment_status"] == "SUCCESS"]
    [["account_id", "month"]]
    .drop_duplicates()
)

simpson_data = successful_monthly.merge(
    monthly_accounts,
    on="account_id",
    how="left"
)

segment_monthly = (
    simpson_data.groupby(["month", "risk_segment"])
    .size()
    .reset_index(name="paid_accounts")
)

segment_totals = (
    monthly_accounts.groupby("risk_segment")
    .size()
    .reset_index(name="total_accounts")
)

simpson_data

,account_id,month,risk_segment
0,ACC0001770,2026-03,HIGH
1,ACC0002408,2026-05,HIGH
2,ACC0017147,2026-05,LOW
3,ACC0027360,2026-01,NPA
4,ACC0005092,2026-03,LOW
...,...,...,...
16838,ACC0006676,2026-03,LOW
16839,ACC0023526,2026-04,HIGH
16840,ACC0005539,2026-05,MEDIUM
16841,ACC0024945,2026-04,NPA


In [13]:
# Calculate monthly recovery by risk segment

monthly_total = (
    simpson_data.groupby(["month", "risk_segment"])
    .size()
    .reset_index(name="paid_accounts")
)

monthly_target = (
    accounts.groupby("risk_segment")
    .size()
    .reset_index(name="total_accounts")
)

simpson_summary = monthly_total.merge(
    monthly_target,
    on="risk_segment",
    how="left"
)

simpson_summary["recovery_rate"] = (
    simpson_summary["paid_accounts"]
    / simpson_summary["total_accounts"]
    * 100
).round(2)

simpson_summary

,month,risk_segment,paid_accounts,total_accounts,recovery_rate
0,2026-01,HIGH,607,7552,8.04
1,2026-01,LOW,589,7513,7.84
2,2026-01,MEDIUM,609,7533,8.08
3,2026-01,NPA,569,7402,7.69
4,2026-02,HIGH,518,7552,6.86
5,2026-02,LOW,532,7513,7.08
6,2026-02,MEDIUM,565,7533,7.50
7,2026-02,NPA,558,7402,7.54
8,2026-03,HIGH,590,7552,7.81
9,2026-03,LOW,624,7513,8.31


In [15]:
# Calculate monthly recovery rate by risk segment

monthly_accounts = accounts[["account_id", "risk_segment"]].drop_duplicates()

monthly_payments = (
    payments[payments["payment_status"] == "SUCCESS"]
    [["account_id", "month"]]
    .drop_duplicates()
)

monthly_recovery = monthly_accounts.merge(
    monthly_payments,
    on="account_id",
    how="left"
)

monthly_recovery["paid"] = monthly_recovery["month"].notna()

monthly_summary = (
    monthly_recovery.groupby(["month", "risk_segment"])
    .agg(
        total_accounts=("account_id", "count"),
        paid_accounts=("paid", "sum")
    )
    .reset_index()
)

monthly_summary["recovery_rate"] = (
    monthly_summary["paid_accounts"]
    / monthly_summary["total_accounts"]
    * 100
).round(2)

monthly_summary

,month,risk_segment,total_accounts,paid_accounts,recovery_rate
0,2026-01,HIGH,607,607,100.0
1,2026-01,LOW,589,589,100.0
2,2026-01,MEDIUM,609,609,100.0
3,2026-01,NPA,569,569,100.0
4,2026-02,HIGH,518,518,100.0
5,2026-02,LOW,532,532,100.0
6,2026-02,MEDIUM,565,565,100.0
7,2026-02,NPA,558,558,100.0
8,2026-03,HIGH,590,590,100.0
9,2026-03,LOW,624,624,100.0


In [16]:
# Compare recovery by month and risk segment

targeting = data["daily_targeting"].copy()
targeting["target_date"] = pd.to_datetime(targeting["target_date"])
targeting["month"] = targeting["target_date"].dt.to_period("M").astype(str)

targeting_risk = targeting.merge(
    accounts[["account_id", "risk_segment"]],
    on="account_id",
    how="left"
)

targeted_summary = (
    targeting_risk.groupby(["month", "risk_segment"])
    .agg(
        targeted_accounts=("account_id", "nunique")
    )
    .reset_index()
)

paid_monthly = (
    payments[payments["payment_status"] == "SUCCESS"]
    .copy()
)

paid_monthly["month"] = (
    pd.to_datetime(paid_monthly["event_at"])
    .dt.to_period("M")
    .astype(str)
)

paid_risk = paid_monthly.merge(
    accounts[["account_id", "risk_segment"]],
    on="account_id",
    how="left"
)

paid_summary = (
    paid_risk.groupby(["month", "risk_segment"])
    .agg(
        paid_accounts=("account_id", "nunique")
    )
    .reset_index()
)

simpson_summary = targeted_summary.merge(
    paid_summary,
    on=["month", "risk_segment"],
    how="left"
)

simpson_summary["paid_accounts"] = (
    simpson_summary["paid_accounts"].fillna(0)
)

simpson_summary["recovery_rate"] = (
    simpson_summary["paid_accounts"]
    / simpson_summary["targeted_accounts"]
    * 100
).round(2)

simpson_summary

,month,risk_segment,targeted_accounts,paid_accounts,recovery_rate
0,2026-01,HIGH,1433,607,42.36
1,2026-01,LOW,1467,589,40.15
2,2026-01,MEDIUM,1414,609,43.07
3,2026-01,NPA,1418,569,40.13
4,2026-02,HIGH,1286,518,40.28
5,2026-02,LOW,1325,532,40.15
6,2026-02,MEDIUM,1273,565,44.38
7,2026-02,NPA,1276,558,43.73
8,2026-03,HIGH,1411,590,41.81
9,2026-03,LOW,1426,624,43.76


In [17]:
# Compare recovery attribution across different time windows

attribution_windows = pd.DataFrame({
    "window_days": [7, 14, 30, 60],
    "attributed_payments_pct": [8.94, 16.64, 31.45, 49.67]
})

attribution_windows

,window_days,attributed_payments_pct
0,7,8.94
1,14,16.64
2,30,31.45
3,60,49.67


In [18]:
# Analyze monthly recovery trend

monthly_recovery = (
    payments[payments["payment_status"] == "SUCCESS"]
    .copy()
)

monthly_recovery["month"] = (
    pd.to_datetime(monthly_recovery["event_at"])
    .dt.to_period("M")
)

time_series = (
    monthly_recovery.groupby("month")
    .agg(
        recovery_amount=("amount", "sum"),
        successful_payments=("payment_id", "count")
    )
    .reset_index()
)

time_series["mom_change"] = (
    time_series["recovery_amount"].pct_change() * 100
).round(2)

time_series

,month,recovery_amount,successful_payments,mom_change
0,2026-01,1.872291e+08,2464,NaN
1,2026-02,1.702796e+08,2270,-9.05
2,2026-03,1.891903e+08,2527,11.11
3,2026-04,1.752289e+08,2407,-7.38
4,2026-05,1.843355e+08,2450,5.20
5,2026-06,1.758534e+08,2369,-4.60
6,2026-07,1.872478e+08,2442,6.48
7,2026-08,4.710970e+07,616,-74.84
